# Notebook 9: Machine Learning Evaluation and Comparison

## Objective

This notebook conducts the final out-of-sample evaluation of the forecasting models developed in Notebook 08.

The analysis uses the locked January–December 2025 test period to:

1. evaluate the persistence, Ridge, Random Forest, and XGBoost forecasts
2. compare symmetric and asymmetric exchange-rate representations
3. examine performance across food subclasses and months
4. identify where forecasting errors are concentrate
5. compare machine-learning forecasts with time-aligned econometric baselines

No models are tuned or selected using the test results.

In [1]:
# import libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

sns.set_theme(style="whitegrid")

print("Evaluation libraries imported successfully.")

Evaluation libraries imported successfully.


In [4]:
# define input and output locations
processed_data_directory = Path("../data/processed")
ml_table_directory = Path("../reports/tables/machine_learning")
econometric_table_directory = Path(
    "../reports/tables/econometrics"
)

evaluation_table_directory = Path(
    "../reports/tables/model_evaluation"
)
evaluation_figure_directory = Path(
    "../reports/figures/model_evaluation"
)

evaluation_table_directory.mkdir(parents=True, exist_ok=True)
evaluation_figure_directory.mkdir(parents=True, exist_ok=True)


# load the locked outcomes and modelling outputs
ml_data = pd.read_csv(
    processed_data_directory / "ml_model_data.csv",
    parse_dates=["Date"],
)

ml_test_predictions = pd.read_csv(
    ml_table_directory / "ml_test_predictions.csv",
    parse_dates=["Date"],
)

validation_model_results = pd.read_csv(
    ml_table_directory / "validation_model_comparison.csv"
)

test_actuals = ml_data.loc[
    ml_data["Split"] == "Test",
    [
        "Date",
        "ClassDescription",
        "SubclassDescription",
        "Food_Inflation_Pct",
    ],
].copy()

print("Machine-learning data loaded:", len(ml_data))
print("Locked test outcomes loaded:", len(test_actuals))
print("Forecast rows loaded:", len(ml_test_predictions))
print("Validation models loaded:", len(validation_model_results))

Machine-learning data loaded: 4554
Locked test outcomes loaded: 552
Forecast rows loaded: 3864
Validation models loaded: 7


In [3]:
# validate the evaluation sample
identifier_columns = [
    "Date",
    "ClassDescription",
    "SubclassDescription",
]

prediction_identifier_columns = identifier_columns + [
    "Model",
    "Representation",
]

prediction_counts = (
    ml_test_predictions.groupby(
        ["Model", "Representation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Predictions"})
)

evaluation_checks = pd.DataFrame(
    {
        "Check": [
            "Test sample contains 552 outcomes",
            "Test sample covers 46 food subclasses",
            "Test sample covers 12 months",
            "Test outcomes contain no duplicate keys",
            "Predictions contain seven model variants",
            "Every model variant contains 552 predictions",
            "Predictions contain no duplicate records",
            "Target was absent from prediction file",
        ],
        "Passed": [
            len(test_actuals) == 552,
            test_actuals["SubclassDescription"].nunique() == 46,
            test_actuals["Date"].nunique() == 12,
            not test_actuals.duplicated(identifier_columns).any(),
            len(prediction_counts) == 7,
            prediction_counts["Predictions"].eq(552).all(),
            not ml_test_predictions.duplicated(
                prediction_identifier_columns
            ).any(),
            "Food_Inflation_Pct"
            not in ml_test_predictions.columns,
        ],
    }
)

if not evaluation_checks["Passed"].all():
    failed_checks = evaluation_checks.loc[
        ~evaluation_checks["Passed"],
        "Check",
    ].tolist()
    raise ValueError(f"Evaluation checks failed: {failed_checks}")


# Attach actual outcomes for final evaluation
forecast_evaluation_data = ml_test_predictions.merge(
    test_actuals,
    on=identifier_columns,
    how="left",
    validate="many_to_one",
)

missing_actuals = int(
    forecast_evaluation_data["Food_Inflation_Pct"]
    .isna()
    .sum()
)

if missing_actuals:
    raise ValueError(
        f"{missing_actuals} predictions could not be matched "
        "to test outcomes."
    )

evaluation_sample_summary = pd.DataFrame(
    {
        "Value": [
            len(test_actuals),
            test_actuals["SubclassDescription"].nunique(),
            test_actuals["Date"].nunique(),
            test_actuals["Date"].min(),
            test_actuals["Date"].max(),
            len(prediction_counts),
            len(forecast_evaluation_data),
            missing_actuals,
        ]
    },
    index=[
        "Test outcomes",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Forecast variants",
        "Forecast-evaluation rows",
        "Missing matched outcomes",
    ],
)

display(evaluation_checks)
display(evaluation_sample_summary)
display(prediction_counts)

print(
    "All evaluation setup checks passed:",
    bool(evaluation_checks["Passed"].all()),
)

,Check,Passed
0,Test sample contains 552 outcomes,True
1,Test sample covers 46 food subclasses,True
2,Test sample covers 12 months,True
3,Test outcomes contain no duplicate keys,True
4,Predictions contain seven model variants,True
5,Every model variant contains 552 predictions,True
6,Predictions contain no duplicate records,True
7,Target was absent from prediction file,True


,Value
Test outcomes,552
Food subclasses,46
Unique months,12
Start date,2025-01-01 00:00:00
End date,2025-12-01 00:00:00
Forecast variants,7
Forecast-evaluation rows,3864
Missing matched outcomes,0


,Model,Representation,Predictions
0,Persistence,Lag-1 benchmark,552
1,Random Forest,Asymmetric,552
2,Random Forest,Symmetric,552
3,Ridge,Asymmetric,552
4,Ridge,Symmetric,552
5,XGBoost,Asymmetric,552
6,XGBoost,Symmetric,552


All evaluation setup checks passed: True


## Overall locked-test performance

The following evaluation compares all seven forecast variants across the complete 2025 test sample.

In addition to MAE, RMSE, R², and directional accuracy, mean bias is reported. A bias is calculated as predicted inflation minus observed inflation. A positive value therefore indicates average overprediction, while a negative value indicates average underprediction.

In [5]:
# calculate overall test metrics
def calculate_forecast_metrics(evaluation_data):
    actual_values = evaluation_data[
        "Food_Inflation_Pct"
    ].to_numpy()

    predicted_values = evaluation_data[
        "Predicted_Food_Inflation_Pct"
    ].to_numpy()

    forecast_errors = predicted_values - actual_values

    return {
        "Observations": len(evaluation_data),
        "MAE": mean_absolute_error(
            actual_values,
            predicted_values,
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual_values,
                predicted_values,
            )
        ),
        "R2": r2_score(
            actual_values,
            predicted_values,
        ),
        "Directional_Accuracy_Pct": (
            np.mean(
                np.sign(actual_values)
                == np.sign(predicted_values)
            )
            * 100
        ),
        "Mean_Bias": np.mean(forecast_errors),
    }


test_metric_records = []

for (
    model_name,
    representation,
), model_data in forecast_evaluation_data.groupby(
    ["Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(model_data)

    test_metric_records.append(
        {
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

ml_test_model_results = pd.DataFrame(test_metric_records)

persistence_result = ml_test_model_results.loc[
    ml_test_model_results["Model"] == "Persistence"
].iloc[0]

ml_test_model_results[
    "RMSE_Improvement_vs_Persistence_Pct"
] = (
    (
        persistence_result["RMSE"]
        - ml_test_model_results["RMSE"]
    )
    / persistence_result["RMSE"]
    * 100
)

ml_test_model_results[
    "MAE_Improvement_vs_Persistence_Pct"
] = (
    (
        persistence_result["MAE"]
        - ml_test_model_results["MAE"]
    )
    / persistence_result["MAE"]
    * 100
)

ml_test_model_results["Test_Rank"] = (
    ml_test_model_results["RMSE"]
    .rank(method="min")
    .astype(int)
)

ml_test_model_results = ml_test_model_results.sort_values(
    ["Test_Rank", "MAE"]
).reset_index(drop=True)

display(
    ml_test_model_results[
        [
            "Test_Rank",
            "Model",
            "Representation",
            "Observations",
            "MAE",
            "RMSE",
            "R2",
            "Directional_Accuracy_Pct",
            "Mean_Bias",
            "RMSE_Improvement_vs_Persistence_Pct",
            "MAE_Improvement_vs_Persistence_Pct",
        ]
    ]
)

leading_test_model = ml_test_model_results.iloc[0]

print("Lowest test RMSE:")
print(
    f"{leading_test_model['Model']} — "
    f"{leading_test_model['Representation']}"
)
print(f"Test RMSE: {leading_test_model['RMSE']:.6f}")

,Test_Rank,Model,Representation,Observations,MAE,RMSE,R2,Directional_Accuracy_Pct,Mean_Bias,RMSE_Improvement_vs_Persistence_Pct,MAE_Improvement_vs_Persistence_Pct
0,1,XGBoost,Symmetric,552,1.060854,1.929075,0.113069,59.782609,0.263793,13.703894,20.423091
1,2,Random Forest,Symmetric,552,1.066689,1.964122,0.080550,63.405797,0.314879,12.136122,19.985359
2,3,XGBoost,Asymmetric,552,1.090656,1.973192,0.072038,61.231884,0.323877,11.730351,18.187517
3,4,Ridge,Symmetric,552,1.089515,1.984985,0.060913,60.869565,0.280788,11.202815,18.273135
4,5,Ridge,Asymmetric,552,1.092858,1.986590,0.059394,61.050725,0.287840,11.131018,18.022346
5,6,Random Forest,Asymmetric,552,1.075816,1.991030,0.055185,61.231884,0.313877,10.932393,19.300687
6,7,Persistence,Lag-1 benchmark,552,1.333117,2.235414,-0.190988,54.347826,0.025886,0.000000,0.000000


Lowest test RMSE:
XGBoost — Symmetric
Test RMSE: 1.929075


In [6]:
# examine the validation-to-test generalisation gap
validation_metrics = validation_model_results.rename(
    columns={
        "MAE": "Validation_MAE",
        "RMSE": "Validation_RMSE",
        "R2": "Validation_R2",
        "Directional_Accuracy_Pct": (
            "Validation_Directional_Accuracy_Pct"
        ),
    }
)

test_metrics = ml_test_model_results.rename(
    columns={
        "MAE": "Test_MAE",
        "RMSE": "Test_RMSE",
        "R2": "Test_R2",
        "Directional_Accuracy_Pct": (
            "Test_Directional_Accuracy_Pct"
        ),
    }
)

validation_test_comparison = validation_metrics.merge(
    test_metrics[
        [
            "Model",
            "Representation",
            "Test_MAE",
            "Test_RMSE",
            "Test_R2",
            "Test_Directional_Accuracy_Pct",
            "Mean_Bias",
            "Test_Rank",
        ]
    ],
    on=["Model", "Representation"],
    how="inner",
    validate="one_to_one",
)

validation_test_comparison[
    "RMSE_Generalisation_Gap"
] = (
    validation_test_comparison["Test_RMSE"]
    - validation_test_comparison["Validation_RMSE"]
)

validation_test_comparison[
    "RMSE_Change_Pct"
] = (
    validation_test_comparison["RMSE_Generalisation_Gap"]
    / validation_test_comparison["Validation_RMSE"]
    * 100
)

validation_test_comparison["Validation_Rank"] = (
    validation_test_comparison["Validation_RMSE"]
    .rank(method="min")
    .astype(int)
)

validation_test_comparison = validation_test_comparison.sort_values(
    "Test_Rank"
).reset_index(drop=True)

display(
    validation_test_comparison[
        [
            "Model",
            "Representation",
            "Validation_Rank",
            "Test_Rank",
            "Validation_RMSE",
            "Test_RMSE",
            "RMSE_Generalisation_Gap",
            "RMSE_Change_Pct",
            "Validation_R2",
            "Test_R2",
            "Validation_Directional_Accuracy_Pct",
            "Test_Directional_Accuracy_Pct",
        ]
    ]
)

,Model,Representation,Validation_Rank,Test_Rank,Validation_RMSE,Test_RMSE,RMSE_Generalisation_Gap,RMSE_Change_Pct,Validation_R2,Test_R2,Validation_Directional_Accuracy_Pct,Test_Directional_Accuracy_Pct
0,XGBoost,Symmetric,1,1,1.722070,1.929075,0.207006,12.020751,0.177449,0.113069,63.949275,59.782609
1,Random Forest,Symmetric,3,2,1.759446,1.964122,0.204676,11.632964,0.141356,0.080550,64.855072,63.405797
2,XGBoost,Asymmetric,2,3,1.728267,1.973192,0.244925,14.171711,0.171518,0.072038,63.405797,61.231884
3,Ridge,Symmetric,5,4,1.869568,1.984985,0.115417,6.173462,0.030509,0.060913,63.768116,60.869565
4,Ridge,Asymmetric,6,5,1.873179,1.986590,0.113410,6.054428,0.026759,0.059394,63.586957,61.050725
5,Random Forest,Asymmetric,4,6,1.796596,1.991030,0.194434,10.822330,0.104713,0.055185,63.949275,61.231884
6,Persistence,Lag-1 benchmark,7,7,2.340242,2.235414,-0.104828,-4.479351,-0.519088,-0.190988,56.340580,54.347826


### Interpretation of overall test performance

The validation-based selection was supported by the locked-test results. Symmetric XGBoost retained first place, with an RMSE of 1.929 and MAE of 1.061. It reduced RMSE by 13.7% and MAE by 20.4% relative to persistence.

All six machine-learning models outperformed persistence and produced positive test R² values. Persistence produced a negative R², indicating that it performed worse than predicting the test-sample mean.

Symmetric Random Forest ranked second by RMSE and achieved the highest
directional accuracy of 63.41%. Therefore, the model that minimised the size of forecast errors was not the model that most frequently predicted the correct sign of food inflation.

Test errors were higher than validation errors for all machine-learning models. For symmetric XGBoost, RMSE increased by approximately 12.0%, while R² declined from 0.177 to 0.113. This deterioration demonstrates the importance of retaining a genuinely unseen test period.

All machine-learning models had positive mean bias, indicating modest average overprediction during 2025. Nevertheless, they remained more accurate than the persistence benchmark.

The symmetric specification achieved a lower overall test RMSE than its
asymmetric counterpart for Ridge, Random Forest, and XGBoost. Subclass-level analysis is required to determine whether asymmetric features nevertheless benefited particular food categories.

In [7]:
# calculate metrics for every model-subclass combination
subclass_metric_records = []

grouping_columns = [
    "ClassDescription",
    "SubclassDescription",
    "Model",
    "Representation",
]

for group_values, group_data in forecast_evaluation_data.groupby(
    grouping_columns,
    sort=False,
):
    (
        class_description,
        subclass_description,
        model_name,
        representation,
    ) = group_values

    metrics = calculate_forecast_metrics(group_data)

    subclass_metric_records.append(
        {
            "ClassDescription": class_description,
            "SubclassDescription": subclass_description,
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

subclass_model_results = pd.DataFrame(subclass_metric_records)

print(
    "Subclass-model results:",
    len(subclass_model_results),
)
print(
    "Expected results:",
    46 * 7,
)
print(
    "Observations per result:",
    sorted(
        subclass_model_results["Observations"].unique()
    ),
)

Subclass-model results: 322
Expected results: 322
Observations per result: [np.int64(12)]


In [8]:
# identify the lowest-RMSE model for each food subclass
subclass_winners = (
    subclass_model_results.sort_values(
        [
            "SubclassDescription",
            "RMSE",
            "MAE",
        ]
    )
    .drop_duplicates(
        subset=["SubclassDescription"],
        keep="first",
    )
    .rename(
        columns={
            "Model": "Winning_Model",
            "Representation": "Winning_Representation",
            "MAE": "Winning_MAE",
            "RMSE": "Winning_RMSE",
            "R2": "Winning_R2",
            "Directional_Accuracy_Pct": (
                "Winning_Directional_Accuracy_Pct"
            ),
        }
    )
    [
        [
            "ClassDescription",
            "SubclassDescription",
            "Winning_Model",
            "Winning_Representation",
            "Winning_MAE",
            "Winning_RMSE",
            "Winning_R2",
            "Winning_Directional_Accuracy_Pct",
        ]
    ]
    .reset_index(drop=True)
)

subclass_winner_summary = (
    subclass_winners.groupby(
        [
            "Winning_Model",
            "Winning_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        "Food_Subclasses",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(subclass_winner_summary)
display(
    subclass_winners.sort_values(
        "Winning_RMSE"
    ).head(10)
)

,Winning_Model,Winning_Representation,Food_Subclasses
0,Random Forest,Symmetric,16
1,XGBoost,Symmetric,7
2,Persistence,Lag-1 benchmark,5
3,Random Forest,Asymmetric,5
4,XGBoost,Asymmetric,5
5,Ridge,Symmetric,4
6,Ridge,Asymmetric,4


,ClassDescription,SubclassDescription,Winning_Model,Winning_Representation,Winning_MAE,Winning_RMSE,Winning_R2,Winning_Directional_Accuracy_Pct
35,Soft drinks,Soft drinks,Persistence,Lag-1 benchmark,0.241626,0.294418,-0.250721,50.000000
0,Other food,Baby food,Persistence,Lag-1 benchmark,0.286480,0.339117,-1.241370,41.666667
12,Fruits and nuts,Fruit and nuts ground and other preparations,XGBoost,Symmetric,0.309381,0.372183,0.355309,75.000000
9,Fish and other seafood,Fish,Random Forest,Symmetric,0.314600,0.383227,-0.051256,75.000000
26,Other food,Other food products n.e.c.,Random Forest,Asymmetric,0.281066,0.408675,-0.220225,75.000000
1,Cereal products,Bread and bakery products,Ridge,Symmetric,0.315104,0.409537,-0.424188,66.666667
42,Vegetables,Vegetables and pulses ground and other prepara...,XGBoost,Symmetric,0.368402,0.434867,-0.154845,58.333333
17,Cereal products,"Macaroni, noodles, couscous and similar pasta ...",XGBoost,Asymmetric,0.348134,0.460431,-0.516696,75.000000
5,"Sugar, confectionery and desserts","Chocolate, cocoa, and cocoa-based food products",XGBoost,Asymmetric,0.394256,0.498729,0.015257,83.333333
44,Water,Water,XGBoost,Symmetric,0.432528,0.524496,-0.036607,66.666667


In [9]:
# compare symmetric and asymmetric RMSE by subclass
ml_subclass_results = subclass_model_results.loc[
    subclass_model_results["Model"] != "Persistence"
].copy()

subclass_representation_comparison = (
    ml_subclass_results.pivot(
        index=[
            "ClassDescription",
            "SubclassDescription",
            "Model",
        ],
        columns="Representation",
        values="RMSE",
    )
    .reset_index()
)

subclass_representation_comparison.columns.name = None

subclass_representation_comparison[
    "Asymmetric_Improvement_Pct"
] = (
    (
        subclass_representation_comparison["Symmetric"]
        - subclass_representation_comparison["Asymmetric"]
    )
    / subclass_representation_comparison["Symmetric"]
    * 100
)

subclass_representation_comparison[
    "Preferred_Representation"
] = np.where(
    subclass_representation_comparison["Asymmetric"]
    < subclass_representation_comparison["Symmetric"],
    "Asymmetric",
    "Symmetric",
)

subclass_representation_summary = (
    subclass_representation_comparison.groupby(
        [
            "Model",
            "Preferred_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        ["Model", "Preferred_Representation"]
    )
    .reset_index(drop=True)
)

display(subclass_representation_summary)

print("Largest asymmetric improvements:")
display(
    subclass_representation_comparison.sort_values(
        "Asymmetric_Improvement_Pct",
        ascending=False,
    ).head(10)
)

,Model,Preferred_Representation,Food_Subclasses
0,Random Forest,Asymmetric,23
1,Random Forest,Symmetric,23
2,Ridge,Asymmetric,11
3,Ridge,Symmetric,35
4,XGBoost,Asymmetric,16
5,XGBoost,Symmetric,30


Largest asymmetric improvements:


,ClassDescription,SubclassDescription,Model,Asymmetric,Symmetric,Asymmetric_Improvement_Pct,Preferred_Representation
96,"Sugar, confectionery and desserts","Chocolate, cocoa, and cocoa-based food products",Random Forest,0.505572,0.575587,12.164121,Asymmetric
129,Vegetables,Vegetables and pulses ground and other prepara...,Random Forest,0.464160,0.525447,11.663939,Asymmetric
12,Cereal products,"Macaroni, noodles, couscous and similar pasta ...",Random Forest,0.498739,0.562427,11.323772,Asymmetric
86,Other food,"Salt, condiments and sauces",XGBoost,0.662000,0.742925,10.892848,Asymmetric
93,Soft drinks,Soft drinks,Random Forest,0.400876,0.445238,9.963738,Asymmetric
65,"Milk, other dairy products and eggs",Other milk and cream,XGBoost,0.739107,0.812784,9.064772,Asymmetric
87,Other food,"Spices, culinary herbs and seeds",Random Forest,0.927382,1.019095,8.999465,Asymmetric
71,Oils and fats,Margarine and similar preparations,XGBoost,0.789302,0.844154,6.497936,Asymmetric
113,Tea,Tea and other plant products for infusion,XGBoost,0.769550,0.816799,5.784706,Asymmetric
63,"Milk, other dairy products and eggs",Other milk and cream,Random Forest,0.785311,0.829602,5.338785,Asymmetric
